# Runtime analysis · Complete research figures

**Deep Learning Beyond Accuracy — BERT / MNLI / HANS**  
A single, reproducible notebook for the complete inference-runtime story: the **original CUDA dynamic W8A8 experiment**, **CPU baseline and tuning**, **CPU operator/memory profiling**, and **three-seed OpenVINO FP32 / INT8 results**.

### Design choices
- Consistent colors for **PyTorch FP32 (blue)**, **TorchAO weight-only INT8 (orange)**, **OpenVINO FP32 (purple)**, **OpenVINO INT8 (teal)**, and **original CUDA W8A8 (red)**.
- The *same FP32 BERT checkpoints* are used across stages. Repeated FP32 throughput measurements reflect different **benchmark settings / runtime backends**, not different trained FP32 models.
- **CUDA plots are separate** from CPU plots. The original CUDA W8A8 and CPU weight-only INT8 are **different quantization configurations**.
- Every completed figure is saved as **PNG (240 dpi)** and **editable SVG**. Calculated summary tables are also saved as CSV.

**Input:** the existing `results/runtime_analysis/` directory (or a separately extracted `runtime_analysis/` directory).  
**Output:** `results/visualization/runtime_comprehensive/` when running in the repo; otherwise a folder beside the extracted data.

Run all notebook cells from top to bottom. No model weights, GPU, OpenVINO installation, or re-benchmarking is needed: **these are visualizations of existing saved results.**

In [ ]:
from pathlib import Path
from collections import OrderedDict
import json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import ticker, patheffects
from matplotlib.colors import Normalize
from IPython.display import display, Markdown

# Optional explicit overrides — usually leave as None.
DATA_ROOT = None        # e.g. Path("../results/runtime_analysis")
OUTPUT_ROOT = None      # e.g. Path("../results/visualization/runtime_comprehensive")
SHOW_FIGURES = True     # Set False to save without displaying each plot
SAVE_FIGURES = True

PALETTE = {
    "torch_fp32": "#355C9A",          # blue
    "torchao_weight_only": "#DB783A", # orange
    "openvino_fp32": "#8360B3",       # purple
    "openvino_int8_sym": "#10947E",   # teal
    "cuda_w8a8": "#C94F62",          # red
}
LABELS = {
    "torch_fp32": "PyTorch FP32",
    "torchao_weight_only": "TorchAO INT8\n(weight-only)",
    "openvino_fp32": "OpenVINO FP32",
    "openvino_int8_sym": "OpenVINO INT8\n(weight-compressed)",
}
ORDER = ["torch_fp32", "torchao_weight_only", "openvino_fp32", "openvino_int8_sym"]
TEXT = "#18283D"
MUTED = "#627086"
GRID = "#E8EDF3"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "font.family": "DejaVu Sans", "font.size": 10.5,
    "axes.titlesize": 15, "axes.titleweight": "bold", "axes.titlecolor": TEXT,
    "axes.labelsize": 10.5, "axes.labelcolor": TEXT,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
    "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "savefig.facecolor": "white", "svg.fonttype": "none",
    "legend.frameon": False, "figure.dpi": 110,
})

def find_runtime_root():
    if DATA_ROOT is not None:
        p = Path(DATA_ROOT).expanduser().resolve()
        if not p.is_dir(): raise FileNotFoundError(f"DATA_ROOT does not exist: {p}")
        return p
    bases = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    candidates = []
    for base in bases:
        candidates += [base / "results" / "runtime_analysis", base / "runtime_analysis",
                       base / "runtime_workspace" / "runtime_analysis"]
    for p in candidates:
        if (p / "experiment3_openvino").is_dir() and (p / "tuning").is_dir():
            return p.resolve()
    raise FileNotFoundError(
        "Can't find runtime_analysis. Extract your results into results/runtime_analysis/ "
        "or set DATA_ROOT manually in the first code cell.")

ROOT = find_runtime_root()
if OUTPUT_ROOT is None:
    OUT = (ROOT.parent / "visualization" / "runtime_comprehensive") if ROOT.parent.name == "results" else (ROOT.parent / "runtime_comprehensive_visuals")
else:
    OUT = Path(OUTPUT_ROOT).expanduser().resolve()
FIG_DIR = OUT / "figures"
TABLE_DIR = OUT / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
SAVED = []

def finish(fig, filename, note=None):
    """Apply footer, save PNG + SVG, display or close."""
    if note:
        fig.text(0.02, 0.015, note, fontsize=8.4, color=MUTED, ha="left", va="bottom")
    if SAVE_FIGURES:
        for suffix in ("png", "svg"):
            fp = FIG_DIR / f"{filename}.{suffix}"
            fig.savefig(fp, dpi=240 if suffix == "png" else None,
                        bbox_inches="tight", pad_inches=0.23, facecolor="white")
        SAVED.append(filename)
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)

def clean_axis(ax, orientation="y"):
    ax.spines["left"].set_visible(orientation != "y")
    ax.spines["bottom"].set_visible(orientation != "x")
    ax.grid(axis=orientation, alpha=0.95)
    ax.tick_params(axis="both", length=0)

def heading(ax, title, subtitle=None):
    ax.set_title(title, loc="left", fontweight="bold", pad=23 if subtitle else 13)
    if subtitle:
        ax.text(0, 1.025, subtitle, transform=ax.transAxes,
                ha="left", va="bottom", color=MUTED, fontsize=9.2)

def save_table(df, name):
    fp = TABLE_DIR / f"{name}.csv"
    df.to_csv(fp, index=False)
    return fp

print(f"Data source : {ROOT}")
print(f"Figure files: {FIG_DIR}")
print(f"CSV tables  : {TABLE_DIR}")

## 1 · Load, validate, and consolidate the benchmark results

We use the **three full-HANS seed runs** for the main four-variant comparison (30,000 HANS examples, 8 CPU threads, batch 32). We load the earlier four-thread/eight-thread full runs separately, and the original CUDA experiment separately. **The 512-example pilot is deliberately excluded from the primary averages.**

In [ ]:
exp3_dir = ROOT / "experiment3_openvino"
raw3 = exp3_dir / "three_seed_results.csv"
if raw3.exists():
    cpu = pd.read_csv(raw3)
else:
    csvs = sorted(exp3_dir.glob("seed*_t8_b32_n30000/benchmark_summary.csv"))
    if not csvs:
        raise FileNotFoundError("Missing Experiment 3 full-HANS results")
    cpu = pd.concat((pd.read_csv(f) for f in csvs), ignore_index=True)

cpu["seed"] = pd.to_numeric(cpu["seed"], errors="raise").astype(int)
cpu = cpu[cpu["variant"].isin(ORDER)].copy()
cpu = cpu[cpu["seed"].isin([42,123,456])].copy()
assert not cpu.duplicated(["seed", "variant"]).any(), "Duplicate seed/variant rows"
assert set(cpu["variant"]) == set(ORDER), "Missing CPU variants"
assert set(cpu["seed"]) == {42,123,456}, "Expected 3 seeds"
for col, expected in [("num_examples",30000),("threads",8),("batch_size",32)]:
    assert (cpu[col] == expected).all(), f"Unexpected configuration in {col}"
assert len(cpu) == 12, f"Expected 12 comparisons; got {len(cpu)}"

cpu["hans_accuracy_pct"] = 100 * cpu["hans_accuracy"].astype(float)
cpu["model"] = pd.Categorical(cpu["variant"], categories=ORDER, ordered=True)
cpu = cpu.sort_values(["model", "seed"])

summary = cpu.groupby("variant", sort=False).agg(
    throughput_mean=("throughput_examples_per_second", "mean"),
    throughput_std=("throughput_examples_per_second", "std"),
    accuracy_mean=("hans_accuracy_pct", "mean"),
    accuracy_std=("hans_accuracy_pct", "std"),
    artifact_mib=("model_artifact_mib", "mean"),
    rss_ready_mean=("rss_ready_mib", "mean"),
    rss_after_mean=("rss_after_inference_mib", "mean"),
    latency_median_ms=("median_batch_forward_ms", "mean"),
    agreement_mean=("three_class_agreement_with_saved_fp32", "mean"),
).reindex(ORDER)
summary["model"] = summary.index.map(lambda m: LABELS[m].replace("\n", " "))
summary["accuracy_delta_vs_fp32_pp"] = summary.accuracy_mean - summary.loc["torch_fp32", "accuracy_mean"]
summary["throughput_ratio_vs_fp32"] = summary.throughput_mean / summary.loc["torch_fp32", "throughput_mean"]
save_table(summary.reset_index(), "cpu_three_seed_summary")

base = cpu.pivot(index="seed", columns="variant", values="throughput_examples_per_second").reindex(columns=ORDER)
paired = base.div(base["torch_fp32"], axis=0)
save_table(paired.reset_index(), "paired_speedups_by_seed")

old4 = pd.read_csv(ROOT / "fp32_vs_weight_only_cpu" / "benchmark_results_n30000.csv")
full8 = pd.read_csv(ROOT / "tuning" / "full_t8_b32" / "benchmark_results_n30000.csv")
old_gpu = pd.read_csv(ROOT / "all_benchmark_variants.csv")
mem2 = pd.read_csv(ROOT / "experiment2_profiler" / "seed42_t8_b32_n256" / "experiment2_memory_timeline.csv")
ops2 = pd.read_csv(ROOT / "experiment2_profiler" / "seed42_t8_b32_n256" / "experiment2_operator_attribution.csv")

# Explicitly check that these are the same seed groups and comparable CPU workload.
for df in (old4, full8):
    assert set(df.seed) == {42,123,456}
    assert set(df.num_examples) == {30000} and set(df.batch_size) == {32}
assert set(old4.threads) == {4} and set(full8.threads) == {8}

print("Experiment 3: 12 configurations = 3 seeds × 4 inference variants")
print("All source files loaded and benchmark settings checked.")
display(summary[["throughput_mean","throughput_std","accuracy_mean","artifact_mib","accuracy_delta_vs_fp32_pp"]].round(3))

## 2 · Main findings — full-HANS, three-seed CPU comparisons

Figures 01–07 compare the **same 30,000 HANS examples per seed**, **8 threads**, **batch size 32**, and **3 timed passes**. Bars show the mean of the three seed-level results; whiskers or dots reveal across-seed variation (not confidence intervals).

In [ ]:
# FIGURE 01 — Overall throughput
fig, ax = plt.subplots(figsize=(11.2,5.8))
xs = np.arange(len(ORDER))
vals = summary.loc[ORDER,"throughput_mean"].to_numpy()
stds = summary.loc[ORDER,"throughput_std"].to_numpy()
ax.bar(xs, vals, width=0.61, color=[PALETTE[k] for k in ORDER], alpha=0.97, zorder=3)
ax.errorbar(xs, vals, yerr=stds, fmt="none", ecolor=TEXT, capsize=5, lw=1.4, zorder=6)
for j, variant in enumerate(ORDER):
    observed = cpu.loc[cpu.variant==variant,"throughput_examples_per_second"].to_numpy()
    ax.scatter(j + np.array([-0.10,0,0.10]), observed, color="white", edgecolor=TEXT,
               s=39, lw=0.85, zorder=7)
    ax.text(j, vals[j]+max(18, stds[j]+12), f"{vals[j]:.1f}", ha="center", color=TEXT, weight="bold", fontsize=12)
ax.set_xticks(xs, [LABELS[k] for k in ORDER]); ax.set_ylim(0, 475)
ax.set_ylabel("HANS examples / second ↑"); clean_axis(ax)
heading(ax, "CPU inference throughput", "All 3 seeds · full HANS · 8 CPU threads · batch 32 · each point = one seed")
fig.tight_layout(rect=[0,0.07,1,0.95])
finish(fig,"01_cpu_throughput", "Bars = mean across seeds; whiskers = sample SD; dots = individual seeds. Higher is faster.")

In [ ]:
# FIGURE 02 — Paired speedups (avoids conflating runs from different seeds)
methods = ["torchao_weight_only", "openvino_fp32", "openvino_int8_sym"]
fig, ax = plt.subplots(figsize=(10.9,5.8))
xx=np.arange(len(methods)); pw=paired[methods]
means=pw.mean(); stds=pw.std(ddof=1)
ax.bar(xx, means, width=.60, color=[PALETTE[k] for k in methods], zorder=3)
ax.errorbar(xx,means,yerr=stds,fmt='none',ecolor=TEXT,lw=1.3,capsize=5,zorder=5)
for i,k in enumerate(methods):
    ax.scatter(i+np.array([-.10,0,.10]),pw[k].to_numpy(),s=43,facecolor='white',edgecolor=TEXT,lw=.85,zorder=6)
    ax.text(i, means[k]+.11,f"{means[k]:.2f}×",ha='center',weight='bold',color=TEXT,fontsize=12)
ax.axhline(1,ls='--',lw=1.25,color=PALETTE['torch_fp32'],alpha=.9)
ax.text(2.54,1.035,"PyTorch FP32 = 1.0×",color=PALETTE['torch_fp32'],ha='right',fontsize=9)
ax.set_xticks(xx,[LABELS[k] for k in methods]);ax.set_ylim(0,3.1);ax.set_ylabel('Throughput relative to paired PyTorch FP32 ↑');clean_axis(ax)
heading(ax,'Paired speedup over PyTorch FP32','Each seed is divided by its own FP32 throughput; mean and SD across 3 seeds')
fig.tight_layout(rect=[0,0.08,1,.95])
finish(fig,'02_paired_speedups','A ratio of 1.0× means no speedup. Seed-level ratios are calculated before averaging.')

In [ ]:
# FIGURE 03 — Seed consistency
fig, ax = plt.subplots(figsize=(10.9,5.9))
for k in ORDER:
    dat=cpu[cpu.variant==k].sort_values('seed')
    ax.plot(dat.seed.astype(str),dat.throughput_examples_per_second,
            marker='o', markersize=9,linewidth=2.5,color=PALETTE[k],label=LABELS[k].replace('\n',' '))
ax.set_ylabel('Examples / second ↑');ax.set_ylim(110,450);clean_axis(ax)
ax.legend(ncol=2,loc='upper center',bbox_to_anchor=(.5,-.12),fontsize=9.4)
heading(ax,'Is the OpenVINO speedup consistent?','Full-HANS throughput by fine-tuning seed · identical 8-thread CPU settings')
ax.annotate('TorchAO seed 123:\nslower than other seeds',xy=('123',float(cpu[(cpu.seed==123)&(cpu.variant=='torchao_weight_only')].throughput_examples_per_second.iloc[0])),
            xytext=(0.52,0.23),textcoords='axes fraction',ha='left',fontsize=9,color=MUTED,
            arrowprops={'arrowstyle':'-', 'color':MUTED, 'lw':1})
fig.tight_layout(rect=[0,.06,1,.95]);finish(fig,'03_seed_consistency','One point = one seed. Between-seed variation also reflects timing variability, not just model weights.')

In [ ]:
# FIGURE 04 — Artifact sizes
fig, ax=plt.subplots(figsize=(10.8,5.7))
size=summary.loc[ORDER,'artifact_mib']
ax.barh(np.arange(4),size.iloc[::-1].to_numpy(),height=.63,
        color=[PALETTE[k] for k in ORDER[::-1]])
for i,k in enumerate(ORDER[::-1]):
    ax.text(float(size[k])+7,i,f"{size[k]:.1f} MiB",va='center',color=TEXT,fontsize=11,weight='bold')
ax.set_yticks(np.arange(4),[LABELS[k] for k in ORDER[::-1]]);ax.set_xlim(0,515)
ax.set_xlabel('Saved model artifact / serialized state (MiB) ↓');clean_axis(ax,orientation='x')
heading(ax,'Quantization substantially reduces storage','PyTorch saved state vs OpenVINO IR artifacts · average sizes across 3 seeds')
fig.tight_layout(rect=[0,.055,1,.95]);finish(fig,'04_artifact_sizes',
    'Storage formats differ between PyTorch and OpenVINO; these are on-disk artifacts, not RAM footprints.')

In [ ]:
# FIGURE 05 — Accuracy vs throughput; no misleading "accuracy is preserved" assertion
fig, ax=plt.subplots(figsize=(11.5,6.1))
for k in ORDER:
    r=summary.loc[k];x=float(r.accuracy_mean);y=float(r.throughput_mean)
    ax.scatter(x,y,s=240,color=PALETTE[k],edgecolor='white',linewidth=2,zorder=4)
    dx,dy={"torch_fp32":(-.023,-24),"torchao_weight_only":(-.02,24),"openvino_fp32":(-.15,24),"openvino_int8_sym":(-.20,24)}[k]
    ax.annotate(LABELS[k].replace('\n',' '),(x,y),xytext=(x+dx,y+dy),fontsize=9.2,color=TEXT,
                arrowprops=dict(arrowstyle='-',lw=.8,color=PALETTE[k]))
ax.set_xlim(summary.accuracy_mean.min()-.35,summary.accuracy_mean.max()+.42)
ax.set_ylim(105,465)
ax.set_xlabel('Mean HANS accuracy across 3 seeds (%) ↑')
ax.set_ylabel('Mean CPU throughput (examples / second) ↑');clean_axis(ax)
heading(ax,'The speed–accuracy tradeoff','Mean values across the same three seeds; a small overall accuracy change can mask subgroup effects')
fig.tight_layout(rect=[0,.055,1,.95]);finish(fig,'05_accuracy_vs_throughput',
    'Overall HANS accuracy is not a direct measure of heuristic robustness; subgroup analysis is still required.')

In [ ]:
# FIGURE 06 — Storage/throughput frontier with accuracy deltas
fig, ax=plt.subplots(figsize=(11.1,6.15))
for k in ORDER:
    r=summary.loc[k];x=float(r.artifact_mib);y=float(r.throughput_mean)
    ax.scatter(x,y,s=240,color=PALETTE[k],edgecolor='white',lw=2,zorder=5)
    positions={"torch_fp32":(430,128),"torchao_weight_only":(205,175),
               "openvino_fp32":(275,224),"openvino_int8_sym":(128,402)}
    d=float(r.accuracy_delta_vs_fp32_pp)
    lab=LABELS[k].replace('\n',' ') + f"\nΔ HANS: {d:+.2f} pp"
    ax.annotate(lab,(x,y),xytext=positions[k],fontsize=9.2,color=TEXT,
                arrowprops={'arrowstyle':'-','lw':.8,'color':PALETTE[k]})
ax.set_xlim(70,550);ax.set_ylim(95,460)
ax.set_xlabel('Serialized artifact size (MiB) ↓');ax.set_ylabel('CPU throughput (examples/s) ↑');clean_axis(ax)
heading(ax,'Size–speed efficiency frontier','Upper-left is smaller and faster; annotations show mean accuracy change vs PyTorch FP32')
fig.tight_layout(rect=[0,.06,1,.95]);finish(fig,'06_size_vs_throughput',
    'On-disk artifact formats differ; points are four inference stacks, not four separately trained BERT models.')

In [ ]:
# FIGURE 07 — CPU process RSS, consistent Experiment-3 measurement definitions
fig,axes=plt.subplots(1,2,figsize=(13,5.2),sharey=True)
for ax,(col,title) in zip(axes,[('rss_ready_mib','Model ready'),('rss_after_inference_mib','After inference')]):
    vals=cpu.groupby('variant')[col].mean().reindex(ORDER)
    xx=np.arange(4)
    ax.bar(xx,vals.to_numpy(),color=[PALETTE[k] for k in ORDER],width=.7,zorder=3)
    for i,v in enumerate(vals):ax.text(i,v+35,f"{v:,.0f}",ha='center',fontsize=10,weight='bold',color=TEXT)
    ax.set_xticks(xx,['PyTorch\nFP32','TorchAO\nINT8','OpenVINO\nFP32','OpenVINO\nINT8'])
    ax.set_ylim(0,2250);ax.set_title(title,loc='left',fontsize=13,pad=13)
    clean_axis(ax)
axes[0].set_ylabel('Mean process RSS (MiB) ↓')
fig.suptitle('CPU process memory depends on the execution backend',x=.05,y=1.04,ha='left',weight='bold',fontsize=16,color=TEXT)
fig.tight_layout(rect=[0,.08,1,.98],w_pad=2.1)
finish(fig,'07_cpu_process_rss',
    'RSS includes interpreter, libraries, runtime allocations, and model; NOT a direct measure of model tensor bytes.')

In [ ]:
# FIGURE 08 — Latency on the full HANS benchmark
fig,ax=plt.subplots(figsize=(10.5,5.5))
vals=summary.loc[ORDER,'latency_median_ms'];xx=np.arange(4)
ax.bar(xx, vals,color=[PALETTE[k] for k in ORDER],width=.62,zorder=3)
for i,v in enumerate(vals): ax.text(i,v+5,f"{v:.1f} ms",ha='center',fontsize=11,weight='bold',color=TEXT)
ax.set_xticks(xx,[LABELS[k] for k in ORDER]);ax.set_ylim(0,260)
ax.set_ylabel('Mean of seed-level median batch latency (ms) ↓');clean_axis(ax)
heading(ax,'Batch-forward latency falls with OpenVINO INT8','Batch size 32 · preprocessed input · includes only model execution, not tokenization')
fig.tight_layout(rect=[0,.08,1,.95]);finish(fig,'08_batch32_forward_latency',
    'Median latency calculated per seed in Experiment 3, then averaged over 3 seeds.')

## 3 · CPU tuning history — why we changed the backend

The original **4-thread** CPU experiment and the tuned **8-thread** experiment are comparisons of the **same PyTorch FP32 and TorchAO weight-only INT8 methods**. The short tuning sweep contains only six measured thread/batch combinations; **blank heatmap cells are unmeasured**, not zeros.

In [ ]:
# FIGURE 09 — Fair full-HANS 4 threads vs 8 threads, from Experiment 1
old_all=pd.concat([old4,full8],ignore_index=True)
fig,ax=plt.subplots(figsize=(10.6,5.8))
xx=np.arange(2);width=.34
for off,k in [(-width/2,'fp32'),(width/2,'weight_only_int8')]:
    vals=old_all[old_all.variant==k].groupby('threads')['median_examples_per_second'].mean().reindex([4,8]).to_numpy()
    color=PALETTE['torch_fp32'] if k=='fp32' else PALETTE['torchao_weight_only']
    ax.bar(xx+off,vals,width=width,color=color,zorder=3,label='PyTorch FP32' if k=='fp32' else 'TorchAO weight-only INT8')
    for i,v in enumerate(vals):ax.text(i+off,v+3,f"{v:.1f}",ha='center',color=TEXT,fontsize=10,weight='bold')
ax.set_xticks(xx,['4 CPU threads','8 CPU threads']);ax.set_ylabel('Full HANS throughput (examples / second) ↑');ax.set_ylim(0,195)
ax.legend(loc='upper left');clean_axis(ax)
heading(ax,'Thread tuning helped both models','Full 30,000 HANS examples × 3 seeds · same models and batch size 32')
fig.tight_layout(rect=[0,.06,1,.95]);finish(fig,'09_cpu_thread_tuning_full',
    'Moving from 4 to 8 threads is a setting change, not an improvement caused by INT8 quantization.')

In [ ]:
# FIGURE 10 — Sparse tuning heatmap (measured configurations only)
tuning_rows=[]
for f in sorted((ROOT/'tuning').glob('t*_b*/benchmark_results_n1000.csv')):
    d=pd.read_csv(f)
    for _,r in d.iterrows():
        tuning_rows.append({'threads':int(r.threads),'batch':int(r.batch_size),
                            'variant':str(r.variant),'examples_per_sec':float(r.median_examples_per_second)})
tuning=pd.DataFrame(tuning_rows)
assert len(tuning)==12, f"Expected 12 tuning rows (6 configurations ×2); got {len(tuning)}"
save_table(tuning,'cpu_tuning_seed42_n1000')
fig,axes=plt.subplots(1,2,figsize=(12.3,5.3),sharex=True,sharey=True)
thread_vals=[2,4,8]; batch_vals=[16,32,64]
for ax,variant,k in zip(axes,['fp32','weight_only_int8'],['torch_fp32','torchao_weight_only']):
    p=tuning[tuning.variant==variant].pivot(index='threads',columns='batch',values='examples_per_sec').reindex(index=thread_vals,columns=batch_vals)
    matrix=np.ma.masked_invalid(p.to_numpy(dtype=float));cmap=plt.get_cmap('YlGnBu').copy();cmap.set_bad('#EFF2F6')
    img=ax.imshow(matrix,cmap=cmap,vmin=20,vmax=170,aspect='auto')
    for yi in range(3):
        for xi in range(3):
            v=p.iloc[yi,xi]
            ax.text(xi,yi,'—' if pd.isna(v) else f'{v:.1f}',ha='center',va='center',
                    color=('white' if pd.notna(v) and v>115 else TEXT),weight='bold',fontsize=11)
    ax.set_xticks(range(3),batch_vals);ax.set_yticks(range(3),thread_vals)
    ax.set_xlabel('Batch size');ax.set_title('PyTorch FP32' if variant=='fp32' else 'TorchAO weight-only INT8',loc='left',fontsize=13,pad=12)
    ax.tick_params(length=0)
axes[0].set_ylabel('CPU threads')
cax=fig.add_axes([.905,.25,.014,.55])
fig.colorbar(img,cax=cax,label='Examples / second ↑')
fig.suptitle('Tuning sweep: thread count matters more than small batch changes',x=.04,y=1.03,ha='left',weight='bold',fontsize=14.5,color=TEXT)
fig.subplots_adjust(left=.09,right=.87,bottom=.13,top=.88,wspace=.18)
finish(fig,'10_tuning_heatmaps',
    'Seed 42, 1,000-example pilot; six tested configurations only. “—” = not measured (no interpolation).')

## 4 · Experiment 2 — Diagnose the original CPU TorchAO path

Profiler measurements come from **seed 42, 256 examples**, and **must not be directly compared to the 30,000-example timing benchmarks**. They show memory lifecycle and operator hotspots, not a proven causal decomposition of the OpenVINO improvement.

In [ ]:
# FIGURE 11 — memory lifecycle
stages=['inputs_ready_before_model','model_ready_after_gc','after_warmup','after_unprofiled_forwards','after_malloc_trim']
names=['Inputs\nready','Model\nready','After\nwarm-up','After\ninference','After\ntrim']
fig,ax=plt.subplots(figsize=(11.1,5.7))
for k,lab,col in [('fp32','PyTorch FP32','torch_fp32'),('weight_only_int8','TorchAO INT8 (weight-only)','torchao_weight_only')]:
    d=mem2[(mem2.variant==k)&(mem2.stage.isin(stages))].set_index('stage').reindex(stages)
    assert not d.rss_mib.isna().any()
    ax.plot(range(5),d.rss_mib,marker='o',ms=8,lw=2.5,color=PALETTE[col],label=lab)
    for x,y in enumerate(d.rss_mib):ax.annotate(f'{y:,.0f}',(x,y),xytext=(0,10 if k=='weight_only_int8' else -18),textcoords='offset points',ha='center',fontsize=9,color=PALETTE[col])
ax.set_xticks(range(5),names);ax.set_ylim(800,1650);ax.set_ylabel('Resident set size (MiB)')
clean_axis(ax);ax.legend(loc='upper left')
heading(ax,'Why model-size savings did not equal RAM savings','Profiler run · seed 42 · 8 threads · 256 examples · memory stages before profiling')
fig.tight_layout(rect=[0,.075,1,.95]);finish(fig,'11_experiment2_memory_lifecycle',
    'The gap narrows largely during warm-up/inference as FP32 RSS rises; malloc_trim accounts for only a later part.')

In [ ]:
# FIGURE 12 — main operator hotspots: each variant's top 7 (no cross-variant operator equivalence implied)
fig,axes=plt.subplots(1,2,figsize=(13.1,6.1))
for ax,k,col,name in zip(axes,['fp32','weight_only_int8'],['torch_fp32','torchao_weight_only'],['PyTorch FP32','TorchAO weight-only INT8']):
    d=ops2[ops2.variant==k].nlargest(7,'self_cpu_time_ms').iloc[::-1].copy()
    ax.barh(np.arange(len(d)),d.self_cpu_time_ms,color=PALETTE[col],height=.68)
    ax.set_yticks(np.arange(len(d)),d.operator.map(
        lambda s: 'SDPA flash attention' if 'scaled_dot_product_flash_attention' in str(s)
        else str(s).replace('aten::','')))
    ax.set_xlabel('Self CPU time (ms; profiled workload)')
    ax.set_title(name,loc='left',fontsize=12.5,pad=12)
    ax.set_xlim(0,max(d.self_cpu_time_ms)*1.19)
    for i,v in enumerate(d.self_cpu_time_ms):ax.text(v+max(d.self_cpu_time_ms)*.015,i,f'{v:,.0f}',va='center',fontsize=9,color=TEXT)
    clean_axis(ax,orientation='x')
fig.suptitle('Operator attribution in the PyTorch CPU path',x=.035,y=1.04,ha='left',weight='bold',fontsize=15.5,color=TEXT)
fig.tight_layout(rect=[0,.075,1,.98],w_pad=3)
finish(fig,'12_experiment2_cpu_operator_hotspots',
    'Self CPU time from one profiler run; accumulated operator time is not end-to-end elapsed time.')

In [ ]:
# FIGURE 13 — conversion/copy overhead: small operators visible on their own scale
selected=['aten::copy_','aten::_to_copy','aten::mul','aten::add_']
piv=ops2[ops2.operator.isin(selected)].pivot(index='operator',columns='variant',values='self_cpu_time_ms').fillna(0).reindex(selected).fillna(0)
fig,ax=plt.subplots(figsize=(10.8,5.55));xx=np.arange(len(piv));w=.37
for off,variant,col,name in [(-w/2,'fp32','torch_fp32','PyTorch FP32'),(w/2,'weight_only_int8','torchao_weight_only','TorchAO weight-only INT8')]:
    vals=piv[variant].to_numpy();ax.bar(xx+off,vals,w,color=PALETTE[col],label=name,zorder=3)
    for i,v in enumerate(vals):
        if v>.05:ax.text(i+off,v+4,f'{v:.1f}',ha='center',color=TEXT,fontsize=9)
ax.set_xticks(xx,[s.replace('aten::','') for s in selected]);ax.set_ylabel('Profiled self CPU time (ms)');ax.set_ylim(0,max(260,float(piv.max().max())*1.25))
clean_axis(ax);ax.legend(loc='upper left')
heading(ax,'Additional conversions and copies in TorchAO','A focused view of smaller operators hidden behind dominant matrix multiplications')
fig.tight_layout(rect=[0,.07,1,.95]);finish(fig,'13_experiment2_copy_conversion',
    'Associations are diagnostic, not proof that any one operator caused the benchmark throughput difference.')

## 5 · Historical CUDA experiment — a separate platform and INT8 configuration

The original result was **NVIDIA GPU, seed 42, TorchAO dynamic W8A8** (both weight and activation quantization). It **must not be put on the same absolute-throughput axes as the CPU experiments** or described as the same weight-only quantization method.

In [ ]:
# FIGURE 14 — Historical GPU throughput and latency: dedicated figure
assert set(old_gpu.variant)=={'fp32_eager','int8_eager'}
gpu=old_gpu.set_index('variant')
fig,axes=plt.subplots(1,2,figsize=(12.3,5.4))
variants=['fp32_eager','int8_eager']; colors=[PALETTE['torch_fp32'],PALETTE['cuda_w8a8']]
labs=['CUDA FP32','CUDA TorchAO\ndynamic W8A8']
for ax,col,title,ylabel in [(axes[0],'median_throughput_examples_per_second','GPU throughput','Examples / second ↑'),
                             (axes[1],'batch32_median_forward_latency_ms','GPU batch-32 forward latency','Milliseconds ↓')]:
    vals=gpu.loc[variants,col].to_numpy(dtype=float)
    ax.bar(np.arange(2),vals,color=colors,width=.62,zorder=3)
    for i,v in enumerate(vals):ax.text(i,v+max(vals)*.035,f'{v:,.1f}',ha='center',weight='bold',color=TEXT,fontsize=11)
    ax.set_xticks(range(2),labs);ax.set_ylim(0,max(vals)*1.25);ax.set_ylabel(ylabel)
    ax.set_title(title,loc='left',pad=14,fontsize=13);clean_axis(ax)
fig.suptitle('Original CUDA W8A8: smaller model but slower inference',x=.035,y=1.05,ha='left',fontsize=15.5,weight='bold',color=TEXT)
fig.tight_layout(rect=[0,.09,1,.98],w_pad=2)
finish(fig,'14_original_cuda_w8a8_performance',
    'SEPARATE NVIDIA GPU experiment, seed 42. Do not compare these absolute throughputs with CPU charts.')

In [ ]:
# FIGURE 15 — Historical GPU memory tradeoff
fig,axes=plt.subplots(1,2,figsize=(12.3,5.4))
for ax,col,title,ylabel in [
    (axes[0],'median_peak_cuda_allocated_mib','Peak allocated CUDA memory','Peak allocated (MiB) ↓'),
    (axes[1],'serialized_state_mib','Serialized model size','Serialized size (MiB) ↓')]:
    vals=gpu.loc[variants,col].to_numpy(dtype=float)
    ax.bar(range(2),vals,color=colors,width=.62,zorder=3)
    for i,v in enumerate(vals):ax.text(i,v+max(vals)*.035,f'{v:,.1f}',ha='center',weight='bold',color=TEXT,fontsize=11)
    ax.set_xticks(range(2),labs);ax.set_ylim(0,max(vals)*1.25);ax.set_ylabel(ylabel)
    ax.set_title(title,loc='left',pad=14,fontsize=13);clean_axis(ax)
fig.suptitle('Original CUDA W8A8: lower GPU memory and model storage',x=.035,y=1.05,ha='left',fontsize=15.5,weight='bold',color=TEXT)
fig.tight_layout(rect=[0,.09,1,.98],w_pad=2)
finish(fig,'15_original_cuda_w8a8_memory',
    'CUDA memory metrics are device allocated bytes, distinct from CPU process RSS and serialized artifact size.')

## 6 · Supplemental reproducibility view

The final figure visualizes each individual timed pass (rather than only medians). This helps identify unstable measurements without inventing confidence intervals from three seeds.

In [ ]:
# FIGURE 16 — repeat-level stability across three seeds, four methods
import ast
rep_rows=[]
for _,r in cpu.iterrows():
    values=r.full_pass_seconds
    if isinstance(values,str): values=ast.literal_eval(values)
    for run_num,seconds in enumerate(values,1):
        rep_rows.append({'seed':int(r.seed),'variant':r.variant,'repeat':run_num,
                         'seconds':float(seconds),'throughput':int(r.num_examples)/float(seconds)})
reps=pd.DataFrame(rep_rows)
save_table(reps,'cpu_full_hans_passes')
fig,axes=plt.subplots(1,3,figsize=(14.2,4.8),sharey=True)
for ax,seed in zip(axes,[42,123,456]):
    d=reps[reps.seed==seed]
    for j,k in enumerate(ORDER):
        vals=d[d.variant==k].sort_values('repeat').throughput.to_numpy()
        ax.scatter(j+np.array([-.10,0,.10]),vals,color=PALETTE[k],s=42,alpha=.9)
        ax.hlines(np.median(vals),j-.17,j+.17,color=TEXT,lw=1.2)
    ax.set_xticks(range(4),['PT\nFP32','AO\nINT8','OV\nFP32','OV\nINT8']);ax.set_xlim(-.4,3.4)
    ax.set_ylim(115,435);ax.set_title(f'Seed {seed}',loc='left',fontsize=12.5,pad=12);clean_axis(ax)
axes[0].set_ylabel('Throughput for each 30k pass (examples/s) ↑')
fig.suptitle('Run-to-run consistency: all three timed passes',x=.037,y=1.05,ha='left',fontsize=15,weight='bold',color=TEXT)
fig.tight_layout(rect=[0,.085,1,.99],w_pad=2.2)
finish(fig,'16_repeat_level_stability',
    'Three dots per variant/seed = separate timed passes; horizontal marker = median of the passes.')

## 7 · Figure index and interpretation notes

The notebook saves figures under `FIG_DIR`, plus consolidated data tables under `TABLE_DIR`. Use **SVG** for slide editing and **PNG** for documents.

**Key interpretation safeguards**
- **One trained FP32 model per seed**, not separate “old” and “new” FP32 models. The PyTorch CPU FP32 result is measured again under different benchmarking settings; OpenVINO FP32 is a different **execution backend** on those same trained weights.
- **TorchAO dynamic W8A8 on CUDA** is not the same configuration as **TorchAO weight-only INT8 on CPU**.
- **NNCF INT8 compressed weights** plus OpenVINO CPU execution is a combined pipeline change. Its speed gain is not attributable exclusively to NNCF or weight precision.
- **Model artifacts ≠ in-memory RAM**. RSS includes process and runtime overhead.
- **Across-seed SD is not an uncertainty interval for runtime deployment.** Three seeds reveal consistency but do not replace controlled performance-repeat methodology.
- **Overall HANS accuracy is insufficient to assess shortcut reliance**. These are runtime figures, not a HANS subgroup/flip analysis.

In [ ]:
index=pd.DataFrame([{'figure':i+1,'basename':name,'PNG':str(FIG_DIR/f'{name}.png'),'SVG':str(FIG_DIR/f'{name}.svg')}
                    for i,name in enumerate(SAVED)])
save_table(index, 'figure_index')
manifest={
    'source_directory':str(ROOT),
    'output_directory':str(OUT),
    'benchmark':{'seeds':[42,123,456],'num_examples':30000,'batch_size':32,'cpu_threads':8},
    'figures':SAVED,
    'notes':'CUDA W8A8 separately plotted. CPU old/new FP32 are same trained checkpoints; timing configurations differ.'
}
(OUT/'README_VISUALS.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print(f'Generated {len(SAVED)} figures × 2 formats = {len(SAVED)*2} plot files')
print(f'Figures -> {FIG_DIR}')
print(f'Tables  -> {TABLE_DIR}')
display(index[['figure','basename']])